# Rough Bergomi Calibration — Final Comparison

This is what the whole project builds toward.

We calibrate Rough Bergomi to the SPX smile and compare all three models side by side:
Black-Scholes, Heston, and Rough Bergomi. One market smile. Three attempts to reproduce it.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from data.loader import load_spx_options
from models.black_scholes import compute_smile
from models.heston import heston_smile
from models.rough_bergomi import rbergomi_smile
from calibration.heston_calibrator import calibrate_heston
from calibration.rbergomi_calibrator import calibrate_rbergomi

In [ ]:
r = 0.05
spot, expiry, T, raw_strikes, raw_prices = load_spx_options(expiry_index=4)
strikes, market_ivols = compute_smile(raw_strikes, raw_prices, spot, T, r)

In [ ]:
# Black-Scholes: flat vol at ATM
atm_idx  = np.argmin(np.abs(strikes - spot))
atm_vol  = market_ivols[atm_idx]
bs_ivols = np.full(len(strikes), atm_vol)
bs_rmse  = np.sqrt(np.mean((bs_ivols - market_ivols) ** 2))

print(f"BS RMSE: {bs_rmse*100:.3f} vol pts (baseline)")

In [ ]:
# Heston calibration
heston_params, heston_rmse = calibrate_heston(spot, strikes, market_ivols, T, r)

heston_ivols = heston_smile(
    spot, strikes, T, r,
    v0=heston_params['v0'], kappa=heston_params['kappa'],
    theta=heston_params['theta'], xi=heston_params['xi'],
    rho=heston_params['rho']
)

In [ ]:
# Rough Bergomi calibration
rb_params, rb_rmse = calibrate_rbergomi(strikes, market_ivols, spot, T, r)

strikes_norm = strikes / spot
rb_ivols = rbergomi_smile(
    strikes_norm, T, r,
    H=rb_params['H'], eta=rb_params['eta'],
    rho=rb_params['rho'], xi0=rb_params['xi0']
)

In [ ]:
# Summary table
print("\n Model Comparison")
print(f" {'Model':<20} {'RMSE (vol pts)':>15}")
print(f" {'-'*36}")
print(f" {'Black-Scholes':<20} {bs_rmse*100:>15.4f}")
print(f" {'Heston':<20} {heston_rmse*100:>15.4f}")
print(f" {'Rough Bergomi':<20} {rb_rmse*100:>15.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(strikes, market_ivols * 100, 'o-',  linewidth=2.5, color='steelblue',  label='Market')
ax.plot(strikes, rb_ivols * 100,     's-',  linewidth=2.5, color='seagreen',   label=f'Rough Bergomi  (RMSE {rb_rmse*100:.3f} vpts)')
ax.plot(strikes, heston_ivols * 100, '^--', linewidth=2,   color='darkorange', label=f'Heston  (RMSE {heston_rmse*100:.3f} vpts)')
ax.plot(strikes, bs_ivols * 100,     ':',   linewidth=1.5, color='tomato',     label=f'Black-Scholes  (RMSE {bs_rmse*100:.3f} vpts)')
ax.axvline(x=spot, color='gray', linestyle=':', alpha=0.5, label='ATM')

ax.set_xlabel('Strike', fontsize=13)
ax.set_ylabel('Implied Volatility (%)', fontsize=13)
ax.set_title(f'BS vs Heston vs Rough Bergomi  |  SPX  |  Expiry {expiry}', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/03_final_comparison.png', dpi=150)
plt.show()

print(f"\nRough Bergomi params:  H={rb_params['H']:.4f}  eta={rb_params['eta']:.4f}  rho={rb_params['rho']:.4f}")

Rough Bergomi captures the short-term skew that both Black-Scholes and Heston miss.

The calibrated H ≈ 0.1 matches the empirical roughness of SPX realized volatility
documented in Gatheral, Jaisson and Rosenbaum (2018). The model reproduces the
steep downward slope of the smile at short maturities with very few parameters.